In [ ]:
import jupyter_black
jupyter_black.load()

import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm import tqdm

from ccl_science_data.common import get_arr, EntC, GenReader

In [ ]:
gr = GenReader("..")

In [ ]:
gr.load_varr_works_citing_sizes().sum()

In [ ]:
gr.load_varr_works_citing_targets().shape

In [ ]:
wyears = larr_work_years()
wcits = larr_work_citing_counts()

In [ ]:
# wtts, wtsis = lvarr_work_topics()

In [ ]:
flat_sfs, wsf_sis = lvarr_work_subfields()

In [ ]:
tsufs = larr_topic_subfields()

In [ ]:
nsfs = tsufs.max() + 1
act_year = wyears.max()

In [ ]:
n_year_dummys = 15

In [ ]:
def get_wsufs():
    wsufs = np.zeros((wtsis.shape[0], nsfs), dtype=np.uint8)
    i = 0
    for wi, wtsize in enumerate(tqdm(wtsis)):
        for j in range(wtsize):
            sufid = tsufs[wtts[i]]
            wsufs[wi, sufid] += 1
            i += 1
    return wsufs

In [ ]:
def get_flats():
    n = flat_sfs.shape[0]
    flat_cites = np.zeros(n, dtype=wcits.dtype)
    flat_year_rates = np.zeros((n, n_year_dummys), dtype=np.float16)

    # flat_sfs = np.zeros(n, dtype=tsufs.dtype)
    i = 0
    for wi, wsf_size in enumerate(tqdm(wsf_sis)):
        for j in range(wsf_size):
            flat_cites[i] = wcits[wi]
            pasy = int(min(act_year - wyears[wi], n_year_dummys * 2 - 1) // 2)
            flat_year_rates[i, pasy] = 4
            for pn, val in [(1, 2), (2, 1)]:
                for mul in [1, -1]:
                    ypi = pasy + mul * pn
                    if ypi >= 0 and ypi < n_year_dummys:
                        flat_year_rates[i, ypi] = val
            i += 1
    return flat_cites, flat_year_rates / flat_year_rates.sum(axis=1).reshape(-1, 1)

In [ ]:
flat_cites, flat_years = get_flats()

In [ ]:
discard_global_top = 0.01
discard_sf_top = 0.01
min_papers = 30
min_cites = 120

In [ ]:
filt_arr = (wcits > 0) & (wcits < np.quantile(wcits, 1 - discard_global_top))

In [ ]:
filt_arr = (flat_cites > 0) & (flat_cites < np.quantile(wcits, 1 - discard_global_top))

In [ ]:
filt_arr.sum() / 1e6

In [ ]:
dropsuf = []
for sfid in tqdm(range(nsfs)):
    sf_farr = flat_sfs == sfid
    if (sf_farr.sum() < min_papers) or (flat_cites[sf_farr].sum() < min_cites):
        dropsuf.append(sfid)
        filt_arr &= ~sf_farr
        continue
    filt_arr &= ~(
        (flat_cites >= np.quantile(flat_cites[sf_farr], 1 - discard_sf_top)) & sf_farr
    )

In [ ]:
filt_arr.sum() / 1e6

In [ ]:
rng = np.random.RandomState()

In [ ]:
valid_inds = np.where(filt_arr)[0]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cpu"
# num_features = wsufs.shape[1]
num_features = tsufs.max() + 1
coeff_dt = torch.float32

In [ ]:
bs = torch.randn(
    (num_features, n_year_dummys), requires_grad=True, device=device, dtype=coeff_dt
)
optimizer = torch.optim.Adam([bs], lr=0.001)

In [ ]:
n = 200_000
maxy = 20

flat_inds = torch.from_numpy(flat_sfs).to(device)
t_wsufs = torch.empty((n, num_features), dtype=torch.float32, device=device)
t_years = torch.empty((n, n_year_dummys), dtype=torch.float32, device=device)
t_wcites = torch.empty((n,), dtype=torch.bfloat16, device=device)

batch_is = [*range(0, valid_inds.shape[0], n)][:-1]
for step in range(40):
    rng.shuffle(valid_inds)
    sumloss = 0
    for i in tqdm(batch_is):
        stinds = valid_inds[i : (i + n)]
        subset_long = torch.from_numpy(flat_sfs[stinds]).to(device).long()
        t_wsufs.zero_().scatter_(1, subset_long.unsqueeze(1), 1.0)
        t_years.copy_(torch.from_numpy(flat_years[stinds, :].astype(np.float32)))
        t_wcites.copy_(
            torch.from_numpy(flat_cites[stinds].astype(np.float32)).to(torch.bfloat16)
        )

        # Training step as before
        optimizer.zero_grad()

        a1 = torch.matmul(t_wsufs, bs)
        a2 = a1 * t_years
        pred = a2.sum(axis=1)
        assert pred.isnan().sum() == 0
        loss = torch.mean((pred - t_wcites) ** 2)
        loss.backward()
        optimizer.step()
        sumloss += loss.item()

    print(step, sumloss / len(batch_is))

In [ ]:
a = torch.randn(num_features, requires_grad=True, device=device, dtype=coeff_dt)
b = torch.randn(num_features, requires_grad=True, device=device, dtype=coeff_dt)
optimizer = torch.optim.Adam([a, b], lr=0.001)

In [ ]:
n = 200_000
maxy = 20

flat_inds = torch.from_numpy(flat_sfs).to(device)
t_wsufs = torch.empty((n, num_features), dtype=torch.float32, device=device)
out_since = torch.empty((n, 1), dtype=torch.float16, device=device)
t_wcites = torch.empty((n,), dtype=torch.bfloat16, device=device)

batch_is = [*range(0, valid_inds.shape[0], n)][:-1]
for step in range(50):
    rng.shuffle(valid_inds)
    sumloss = 0
    for i in tqdm(batch_is):
        stinds = valid_inds[i : (i + n)]
        subset_long = torch.from_numpy(flat_sfs[stinds]).to(device).long()
        t_wsufs.zero_().scatter_(1, subset_long.unsqueeze(1), 1.0)
        out_since.copy_(
            torch.from_numpy(
                (act_year - flat_years[stinds]).astype(np.float16).clip(min=1, max=maxy)
            ).view(-1, 1)
        )
        t_wcites.copy_(
            torch.from_numpy(flat_cites[stinds].astype(np.float32)).to(torch.bfloat16)
        )

        # Training step as before
        optimizer.zero_grad()
        beta = torch.sigmoid(b)
        # alpha = torch.exp(a)
        alpha = torch.nn.functional.softplus(a)
        pred = (torch.matmul(t_wsufs, alpha.view(-1, 1)) * out_since) ** torch.matmul(
            t_wsufs, beta.view(-1, 1)
        )
        assert pred.isnan().sum() == 0
        loss = torch.mean((pred - t_wcites.view(-1, 1)) ** 2)
        loss.backward()
        optimizer.step()
        sumloss += loss.item()

    print(step, sumloss / len(batch_is))

In [ ]:
t_wcites

In [ ]:
wcits[stinds].max(), t_wcites.max()

In [ ]:
_sdf = (
    pd.Series((pred.view(-1) - t_wcites).detach().cpu().numpy(), name="miss")
    .to_frame()
    .assign(
        y=out_since.detach().cpu().numpy(),
        pred=pred.detach().cpu().numpy(),
        # cites=wcits[stinds],
        cites=flat_cites[stinds],
    )
    .astype(float)
)
_sdf.groupby("y").agg("mean").loc[:35, :].plot()

In [ ]:
_sdf.groupby("y").count()["pred"].loc[1:,].plot()

In [ ]:
_sdf.groupby("y").agg("std").loc[:35, :].plot()

In [ ]:
from ccl_science_data.common import GenReader, get_arr

In [ ]:
gr = GenReader("..")

In [ ]:
recs = []
j = 0
for i, name in enumerate(gr.get_names(EntC.SUBFIELDS)):
    if i in dropsuf:
        recs.append({"name": name})
        continue
    recs.append({"name": name, "decay": float(beta[j]), "volume": float(alpha[j])})
    j += 1

In [ ]:
pdf = pd.DataFrame(recs).sort_values("volume", ascending=False)

In [ ]:
pdf.head()

In [ ]:
pdf.tail(30)

In [ ]:
pdf.set_index("name").corr()